<H1> Lab 14 </H1>

# Lab 4: Convolutional Neural Networks (CNNs) and Image Loading -

**Objective:** In previous labs, we loaded datasets directly into memory using PyTorch's built-in datasets. In the real world, you will usually receive a hard drive or folder full of images. In this lab, you will:
1. Load image files directly from a disk directory structure.
2. Build a Convolutional Neural Network (CNN) to process 3-channel (RGB) color images.
3. Compare the spatial feature extraction of CNNs to the MLPs used in previous labs.

In [1]:
import torch
import os
import torchvision
from PIL import Image
from tqdm import tqdm

def setup_disk_dataset(root_dir='./data/cifar10_disk'):
    if os.path.exists(root_dir):
        print("Dataset already exists on disk!")
        return

    print("Saving CIFAR-10 images to disk folder structure...")
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)

    for i, (img, label) in enumerate(tqdm(trainset)):
        class_name = trainset.classes[label]
        save_dir = os.path.join(root_dir, 'train', class_name)
        os.makedirs(save_dir, exist_ok=True)
        img.save(os.path.join(save_dir, f'img_{i}.png'))
    print(f"Images saved successfully to {root_dir}/train!")

setup_disk_dataset()

Saving CIFAR-10 images to disk folder structure...


100%|██████████| 170M/170M [00:05<00:00, 32.2MB/s]
100%|██████████| 50000/50000 [00:23<00:00, 2133.11it/s]

Images saved successfully to ./data/cifar10_disk/train!


### Task 1: Define Data Transformations
When loading images from disk, they are read as PIL Images. We need to convert them to PyTorch Tensors and normalize them.

**Task:** Use `torchvision.transforms.Compose` to convert the images to tensors and normalize them with a mean and standard deviation of 0.5 across all three RGB channels.

In [9]:
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

### Task 2: Load Dataset from Disk
PyTorch provides `torchvision.datasets.ImageFolder` to easily load images that are organized in folders named after their classes.

**Task:** Use `ImageFolder` to load the training dataset from the `./data/cifar10_disk/train` directory, applying the transform you defined in Task 1.

In [10]:
from torchvision.datasets import ImageFolder

train_dataset = ImageFolder(root='./data/cifar10_disk/train',transform = transform)

print(f"Loaded {len(train_dataset)} images.")
print(f"classes: {train_dataset.classes}")

Loaded 50000 images.
classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [ ]:
import matplotlib.pyplot as plt

### Task 3: Create the DataLoader
**Task:** Wrap your `train_dataset` in a `torch.utils.data.DataLoader`. Set a batch size of 64 and ensure the data is shuffled.

In [11]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset,batch_size=64,shuffle=True)

### Task 4: Build the CNN Architecture


Unlike MLPs that flatten the image immediately, CNNs use convolutional layers to extract spatial features.

**Task:** Create a class `SimpleCNN` that inherits from `nn.Module`.
Your network should have:
1. A convolutional layer (`nn.Conv2d`) taking 3 input channels, outputting 16 channels, with a kernel size of 3.
2. A ReLU activation.
3. A Max Pooling layer (`nn.MaxPool2d`) with a kernel size of 2 and stride of 2.
4. A fully connected layer (`nn.Linear`) that maps the flattened pooled features to 10 output classes. *(Hint: CIFAR-10 images are 32x32. Calculate the spatial dimensions after convolution and pooling to find the input size for the Linear layer).*

In [12]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # 1. Conv layer: 3 input -> 16 output channels, kernel 3
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3)

        # 2. Max Pool layer: kernel 2, stride 2
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # 3. Fully connected layer: maps flattened features to 10 classes
        # Calculation: (32-3+1) = 30 -> (30/2) = 15. Linear input = 16 * 15 * 15
        self.fc1 = nn.Linear(16 * 15 * 15, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = torch.flatten(x, 1) # Flatten all dimensions except batch
        x = self.fc1(x)
        return x

    #initialize the model
model = SimpleCNN()
print(model)

SimpleCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=3600, out_features=10, bias=True)
)


### Task 5: Define Loss and Optimizer
**Task:** Initialize the Cross-Entropy Loss function and the Adam optimizer with a learning rate of 0.001.

In [13]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

### Task 6: The Training Loop
**Task:** Write a training loop for 5 epochs. Iterate over the `train_loader`, perform the forward pass, calculate the loss, backpropagate, and update the weights. Print the loss at the end of each epoch.

In [14]:
epoch =5
for epoch in range(epoch):
    running_loss = 0.0
    for i, data in enumerate(train_loader, 0):
        inputs, labels = data

        #zero the parameter gradiant
        optimizer.zero_grad()

        #forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        #backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    print(f"epoch {epoch +1} ,loss {running_loss / len(train_loader):.4f}")

print("Finished Training")

epoch 1 ,loss 1.4644
epoch 2 ,loss 1.2174
epoch 3 ,loss 1.1218
epoch 4 ,loss 1.0533
epoch 5 ,loss 1.0092
Finished Training


# Test

Files already downloaded and verified


Accuracy of the network on the 10000 test images: 60.42 %
